In [8]:
import pandas as pd
import sqlite3

con = sqlite3.connect('../data/checking-logs.sqlite')

In [9]:
test_results = pd.read_sql_query("""
SELECT
    period AS time,
    AVG(diff) AS avg_diff
FROM (
    SELECT
        t.uid,
        CASE
            WHEN unixepoch(t.first_commit_ts) < unixepoch(t.first_view_ts)
            THEN 'before'
            ELSE 'after'
        END AS period,
        (unixepoch(t.first_commit_ts) - d.deadlines) / 3600.0 AS diff
    FROM test t
    JOIN deadlines d
      ON t.labname = d.labs
    WHERE t.labname <> 'project1'
) base
WHERE uid IN (
    SELECT uid
    FROM (
        SELECT
            t.uid,
            COUNT(DISTINCT
                CASE
                    WHEN unixepoch(t.first_commit_ts) < unixepoch(t.first_view_ts)
                    THEN 'before'
                    ELSE 'after'
                END
            ) AS periods
        FROM test t
        WHERE t.labname <> 'project1'
        GROUP BY t.uid
    )
    WHERE periods = 2
)
GROUP BY period
ORDER BY time
""", con)

test_results

,time,avg_diff
0,after,-105.229241
1,before,-61.156632


In [10]:
control_results = pd.read_sql_query("""
SELECT
    period AS time,
    AVG(diff) AS avg_diff
FROM (
    SELECT
        c.uid,
        CASE
            WHEN unixepoch(c.first_commit_ts) < unixepoch(c.first_view_ts)
            THEN 'before'
            ELSE 'after'
        END AS period,
        (unixepoch(c.first_commit_ts) - d.deadlines) / 3600.0 AS diff
    FROM control c
    JOIN deadlines d
      ON c.labname = d.labs
    WHERE c.labname <> 'project1'
) base
WHERE uid IN (
    SELECT uid
    FROM (
        SELECT
            c.uid,
            COUNT(DISTINCT
                CASE
                    WHEN unixepoch(c.first_commit_ts) < unixepoch(c.first_view_ts)
                    THEN 'before'
                    ELSE 'after'
                END
            ) AS periods
        FROM control c
        WHERE c.labname <> 'project1'
        GROUP BY c.uid
    )
    WHERE periods = 2
)
GROUP BY period
ORDER BY time
""", con)

control_results

,time,avg_diff
0,after,-118.144571
1,before,-99.901448


In [11]:
con.close()